In [1]:
# === CELL 1: Imports and Setup ===
"""
Feature Build Notebook

This notebook:
1. Loads cleansed 1-minute data
2. Aggregates to decision bars
3. Computes base and rolling features
4. Generates labels
5. Persists feature dataset

Outputs:
- /data/model_data/{symbol}/bars_20m_features.parquet
- /data/model_data/{symbol}/feature_metadata.json
"""
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm import tqdm

# Robust project root discovery
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
from utils import load_config, config_hash
from transformers.bar_aggregator import DecisionBarAggregator
from transformers.feature_pipeline import BaseFeatureExtractor, RollingFeatureExtractor, LagFeatureExtractor, ExpandingFeatureExtractor


In [2]:
# === CELL 2: Load Configuration ===
download_config = load_config(ROOT / "config" / "download.yaml")
pipeline_config = load_config(ROOT / "config" / "pipeline.yaml")
model_config = load_config(ROOT / "config" / "model.yaml")

SYMBOL = download_config["symbol"]
N = pipeline_config["decision_interval"]
WINDOWS = pipeline_config["windows"]
ROLLING_STATS = pipeline_config["rolling_stats"]
ROLLING_BASE = pipeline_config["rolling_base_features"]
LAGS = pipeline_config.get("lags", [])
LAG_FEATURES = pipeline_config.get("lag_features", [])
EXPANDING_STATS = pipeline_config.get("expanding_stats", [])
EXPANDING_FEATURES = pipeline_config.get("expanding_features", [])
BURN_IN = pipeline_config["burn_in_bars"]
TRAIN_FRAC = model_config["train_fraction"]

CLEANSED_PATH = ROOT / download_config["paths"]["cleansed_data"] / SYMBOL
MODEL_DATA_PATH = ROOT / "data" / "model_data" / SYMBOL
MODEL_DATA_PATH.mkdir(parents=True, exist_ok=True)

print(f"Decision interval: {N} minutes")
print(f"Rolling windows: {WINDOWS}")


Decision interval: 20 minutes
Rolling windows: [1, 2, 4, 8, 12, 24, 48, 96]


In [3]:
# === CELL 3: Load Minute Data ===
minute_df = pd.read_parquet(CLEANSED_PATH / "1m.parquet")
print(f"Loaded {len(minute_df):,} minute bars")


Loaded 1,578,160 minute bars


In [4]:
# === CELL 4: Aggregate to Decision Bars ===
aggregator = DecisionBarAggregator(n=N)
decision_bars = []

for _, row in tqdm(minute_df.iterrows(), total=len(minute_df), desc="Aggregating"):
    minute_bar = row.to_dict()
    bar = aggregator.update(minute_bar)
    if bar is not None:
        decision_bars.append(bar)

bars_df = pd.DataFrame(decision_bars)
print(f"Decision bars: {len(bars_df):,}")

bars_df["bar_in_segment"] = bars_df.groupby("segment_id").cumcount()


Aggregating: 100%|██████████| 1578160/1578160 [00:26<00:00, 59767.01it/s]


Decision bars: 78,908


In [5]:
# === CELL 5: Compute Labels ===
bars_df["next_high"] = bars_df["high"].shift(-1)
bars_df["next_segment_id"] = bars_df["segment_id"].shift(-1)
bars_df["log_excursion"] = np.where(
    bars_df["segment_id"] == bars_df["next_segment_id"],
    np.log(bars_df["next_high"] / bars_df["close"]),
    np.nan,
)

train_end_idx = int(len(bars_df) * TRAIN_FRAC)
calibration_df = bars_df.iloc[: max(train_end_idx - 1, 0)]
train_excursions = calibration_df["log_excursion"].dropna()
if train_excursions.empty:
    raise RuntimeError("No valid excursions for alpha calibration (check gaps, date range, and split fractions).")

if pipeline_config["barrier"]["method"] == "quantile":
    quantile = pipeline_config["barrier"]["quantile"]
    alpha = train_excursions.quantile(quantile)
else:
    alpha = pipeline_config["barrier"]["fixed_alpha"]

bars_df["label"] = np.where(
    bars_df["log_excursion"].notna(),
    (bars_df["log_excursion"] >= alpha).astype(int),
    np.nan,
)

print(f"Barrier alpha: {alpha:.6f} (log-return = {100*(np.exp(alpha)-1):.3f}%)")
print(f"Positive rate (train calibration subset): {train_excursions.ge(alpha).mean():.3%}")


Barrier alpha: 0.004111 (log-return = 0.412%)
Positive rate (train calibration subset): 10.002%


In [6]:
# === CELL 6: Compute Features ===
base_extractor = None
rolling_extractor = None
lag_extractor = None
expanding_extractor = None

features_list = []
prev_segment_id = None
for _, row in tqdm(bars_df.iterrows(), total=len(bars_df), desc="Computing features"):
    bar = row.to_dict()

    if prev_segment_id is None or bar.get("segment_id") != prev_segment_id:
        base_extractor = BaseFeatureExtractor()
        rolling_extractor = RollingFeatureExtractor(
            base_features=ROLLING_BASE,
            windows=WINDOWS,
            stats_to_compute=ROLLING_STATS
        )
        lag_extractor = LagFeatureExtractor(
            base_features=LAG_FEATURES,
            lags=LAGS
        )
        expanding_extractor = ExpandingFeatureExtractor(
            base_features=EXPANDING_FEATURES,
            stats_to_compute=EXPANDING_STATS
        )
        prev_segment_id = bar.get("segment_id")

    base_feats = base_extractor.transform_one(bar)
    base_extractor.learn_one(bar)

    lag_feats = lag_extractor.transform_one(base_feats)
    lag_extractor.learn_one(base_feats)

    rolling_extractor.learn_one(base_feats)
    rolling_feats = rolling_extractor.transform_one(base_feats)

    expanding_extractor.learn_one(base_feats)
    expanding_feats = expanding_extractor.transform_one(base_feats)

    features_list.append({**base_feats, **lag_feats, **rolling_feats, **expanding_feats})

features_df = pd.DataFrame(features_list)
print(f"Feature columns: {len(features_df.columns)}")


Computing features: 100%|██████████| 78908/78908 [02:17<00:00, 573.20it/s]


Feature columns: 726


In [7]:
# === CELL 7: Combine and Clean ===
full_df = pd.concat([
    bars_df[["open_time", "close_time", "close", "segment_id", "bar_in_segment", "label"]].reset_index(drop=True),
    features_df.reset_index(drop=True)
], axis=1)

full_df = full_df.dropna(subset=["label"])
full_df["label"] = full_df["label"].astype(int)

full_df = full_df[full_df["bar_in_segment"] >= BURN_IN].reset_index(drop=True)
print(f"Final rows (after burn-in): {len(full_df):,}")


Final rows (after burn-in): 78,714


In [8]:
# === CELL 8: Feature Sanity Checks ===
print("=== Sanity Checks ===")

sample_idx = 100
if len(full_df) > sample_idx:
    print(f"Sample return check at idx {sample_idx}:")
    print(f"  log_close[{sample_idx}]: {full_df.loc[sample_idx, 'log_close']:.6f}")
    print(f"  return[{sample_idx}]: {full_df.loc[sample_idx, 'return']:.6f}")

print(f"Rolling mean convergence:")
print(f"  return_rolling_mean_12 mean: {full_df['return_rolling_mean_12'].mean():.6f}")
print(f"  return mean: {full_df['return'].mean():.6f}")


=== Sanity Checks ===
Sample return check at idx 100:
  log_close[100]: 9.718992
  return[100]: -0.000209
Rolling mean convergence:
  return_rolling_mean_12 mean: 0.000021
  return mean: 0.000021


In [9]:
# === CELL 9: Save Dataset ===
id_cols = ["open_time", "close_time", "close", "segment_id", "bar_in_segment", "label"]
feature_cols = [c for c in full_df.columns if c not in id_cols]

output_path = MODEL_DATA_PATH / "bars_20m_features.parquet"
full_df.to_parquet(output_path, index=False, engine="pyarrow")

feature_metadata = {
    "alpha": float(alpha),
    "barrier_method": pipeline_config["barrier"]["method"],
    "decision_interval": N,
    "train_fraction": float(TRAIN_FRAC),
    "windows": WINDOWS,
    "rolling_stats": ROLLING_STATS,
    "rolling_base_features": ROLLING_BASE,
    "lags": LAGS,
    "lag_features": LAG_FEATURES,
    "expanding_stats": EXPANDING_STATS,
    "expanding_features": EXPANDING_FEATURES,
    "feature_names": feature_cols,
    "n_features": len(feature_cols),
    "n_samples": len(full_df),
    "positive_rate": float(full_df["label"].mean()),
    "config_hash": config_hash(pipeline_config),
}
pd.Series(feature_metadata).to_json(MODEL_DATA_PATH / "feature_metadata.json")

print(f"Saved to: {output_path}")
print(f"Features: {len(feature_cols)}")
print(f"Samples: {len(full_df):,}")
print(f"Positive rate: {full_df['label'].mean():.3%}")


Saved to: C:\Users\vitil\OneDrive\Desktop\online_barrier_classifier\data\model_data\BTCUSDT\bars_20m_features.parquet
Features: 726
Samples: 78,714
Positive rate: 9.886%
